In [1]:
import json, math
from datetime import datetime
from pathlib import Path
from sentence_transformers import SentenceTransformer

NODES_PATH = Path("nodes.json")
EMBS_PATH  = Path("embeddings.json")

MODEL_NAME = "all-MiniLM-L6-v2"
TOP_K = 10

HALF_LIFE_HOURS = 24.0
W_RECENCY = 1.0
W_IMPORTANCE = 1.0
W_RELEVANCE = 1.0
REL_MIN = 0.62

TYPE_W = {"thought": 1.0, "chat": 1.0, "event": 1.0}

def parse_dt(s: str) -> datetime:
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M"):
        try:
            return datetime.strptime(str(s), fmt)
        except:
            pass
    return datetime.fromisoformat(str(s))

def cosine(a, b) -> float:
    dot = sum(x*y for x, y in zip(a, b))
    na = math.sqrt(sum(x*x for x in a))
    nb = math.sqrt(sum(y*y for y in b))
    return dot/(na*nb) if na and nb else 0.0

def recency(created: datetime, now: datetime) -> float:
    dt_h = max(0.0, (now - created).total_seconds() / 3600.0)
    return 2 ** (-dt_h / HALF_LIFE_HOURS)

def importance(poignancy) -> float:
    p = float(poignancy or 0.0)
    return min(1.0, max(0.0, p / 10.0))

def load_nodes():
    data = json.loads(NODES_PATH.read_text(encoding="utf-8"))
    nodes = list(data.values()) if isinstance(data, dict) else data
    for n in nodes:
        n["_id"] = n.get("node_count") or n.get("node_id")
        n["_type"] = n.get("type")
        n["_t"] = parse_dt(n.get("created"))
        n["_txt"] = (n.get("description") or n.get("text") or "").strip()
        n["_k"] = (n.get("embedding_key") or n.get("description") or n.get("text") or "").strip()
    nodes.sort(key=lambda x: x["_t"])
    return nodes

def load_embs():
    e = json.loads(EMBS_PATH.read_text(encoding="utf-8"))
    return {k: v for k, v in e.items() if isinstance(v, list)}

def retrieve(nodes, embs, query: str, model, now=None, top_k=TOP_K):
    now = now or nodes[-1]["_t"]
    qvec = model.encode([query], normalize_embeddings=True)[0].tolist()

    out = []
    for n in nodes:
        vec = embs.get(n["_k"])
        if vec is None:
            continue

        rel = (cosine(qvec, vec) + 1) / 2
        if rel < REL_MIN:
            continue

        r = recency(n["_t"], now)
        imp = importance(n.get("poignancy"))
        tw = TYPE_W.get(n["_type"], 1.0)

        score = tw * (W_RECENCY * r + W_IMPORTANCE * imp + W_RELEVANCE * rel)
        out.append((score, r, imp, rel, n))

    out.sort(key=lambda x: x[0], reverse=True)
    return out[:top_k]

def build_llm_prompt(query: str, mem_rows):
    mem_lines = []
    for score, r, imp, rel, n in mem_rows:
        mem_lines.append(
            f"- id={n['_id']} type={n['_type']} time={n.get('created')} "
            f"(score={score:.3f}, imp={imp:.2f}, rel={rel:.2f}) :: {n['_txt']}"
        )

    return f"""You are a life-log assistant.

QUESTION:
{query}

RULES:
- Use ONLY the memories below.
- If evidence is insufficient, say exactly: Not enough evidence.
- Always cite evidence IDs in the Evidence section.

MEMORIES (retrieved):
{chr(10).join(mem_lines)}

OUTPUT FORMAT:
Answer: <2-5 sentences>
Evidence:
- id=<...>: <short quote or paraphrase>
- id=<...>: <short quote or paraphrase>
"""

if __name__ == "__main__":
    nodes = load_nodes()
    embs = load_embs()
    model = SentenceTransformer(MODEL_NAME)
    now = nodes[-1]["_t"]

    QUESTIONS = [
    "Who invited Klaus to the Valentine's Day party?",
    "When is the Valentine's Day party scheduled?",
    "What did Klaus and Maria talk about recently?",
    "What role does Hobbs Cafe play in Klaus's daily life?",
    "What does Klaus usually do in the evening?",  # <- your own
]

for i, QUESTION in enumerate(QUESTIONS, 1):
    mem_rows = retrieve(nodes, embs, QUESTION, model, now=now, top_k=TOP_K)
    prompt = build_llm_prompt(QUESTION, mem_rows)

    print("\n" + "=" * 90)
    print(f"PROMPT {i}/5 (copy-paste into ChatGPT/Gemini)")
    print("=" * 90)
    print(prompt)

c:\Users\srina\projects\Generative Agents and Memory-Based Retrieval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 798.50it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



PROMPT 1/5 (copy-paste into ChatGPT/Gemini)
You are a life-log assistant.

QUESTION:
Who invited Klaus to the Valentine's Day party?

RULES:
- Use ONLY the memories below.
- If evidence is insufficient, say exactly: Not enough evidence.
- Always cite evidence IDs in the Evidence section.

MEMORIES (retrieved):
- id=525 type=thought time=2023-02-13 20:07:40 (score=2.352, imp=0.70, rel=0.76) :: Klaus Mueller has close relationships with multiple people
- id=435 type=thought time=2023-02-13 17:47:20 (score=2.348, imp=0.70, rel=0.81) :: Klaus Mueller is invited to Isabella Rodriguez's Valentine's Day party at Hobbs Cafe
- id=479 type=event time=2023-02-13 19:12:10 (score=2.347, imp=0.80, rel=0.68) :: Klaus Mueller is conversing about Maria and Klaus are conversing about their research papers on the applications of quantum physics in computer science and their plans to discuss it over lunch tomorrow at Hobbs Cafe, along with Klaus sharing his progress on quantum computing and Maria express